# Setup

In [1]:
import os
import json
import cv2
import shutil
import subprocess
import torch
import torch.nn as nn
import numpy as np
import urllib.request
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from PIL import Image
from torchvision import transforms, models
from ultralytics import YOLO

# --- CONFIGURATION ---
# Define Project Root (Assuming notebook is in experiments/notebooks/)
def get_project_root() -> Path:
    current_path = Path.cwd()
    for parent in [current_path] + list(current_path.parents):
        if (parent / "data").exists():
            return parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "DATASET" / "SixClassBoxingVIDataset" / "V1"
MODELS_DIR = PROJECT_ROOT / "models" / "yolov11x-pose"

# Select a video to process (Change this name as needed)
VIDEO_FILENAME = "v1.mp4" 
VIDEO_PATH = DATA_DIR / VIDEO_FILENAME

# Paths for generated files
PROCESSED_VIDEO_PATH = DATA_DIR / f"{Path(VIDEO_FILENAME).stem}_30fps.mp4"
FRAMES_DIR = DATA_DIR / "frames_30fps"
TRACKED_JSON = DATA_DIR / "yolo_annotations_tracked.json"
FINAL_JSON = DATA_DIR / "yolo_annotations.json"
CLIPS_DIR = DATA_DIR / "clips_annotated"

# Create directories
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"🎥 Target Video: {VIDEO_PATH}")

📂 Project Root: d:\Repositories\Boxer-Pose-Estimation
🎥 Target Video: d:\Repositories\Boxer-Pose-Estimation\data\DATASET\SixClassBoxingVIDataset\V1\v1.mp4


# Resampling

In [ ]:
def resample_video_30fps(input_path, output_path):
    if output_path.exists():
        print(f"✅ 30 FPS Video already exists: {output_path}")
        return

    print(f"⚙️ Resampling {input_path.name} to strict 30 FPS CFR...")
    
    # -r 30 sets frame rate
    # -vsync cfr forces Constant Frame Rate (prevents drift)
    cmd = [
        "ffmpeg", "-y",
        "-i", str(input_path),
        "-r", "30",
        "-vsync", "cfr",
        str(output_path)
    ]
    
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode == 0:
        print("✅ Conversion Complete.")
    else:
        print("❌ FFmpeg failed:")
        print(result.stderr.decode())

resample_video_30fps(VIDEO_PATH, PROCESSED_VIDEO_PATH)

# Frame Extraction

In [ ]:
def extract_frames_in_ranges(video_path, json_path, output_dir):
    if not json_path.exists():
        print(f"❌ Annotations file not found: {json_path}")
        return

    print(f"📖 Loading ranges from {json_path.name}...")
    with open(json_path, 'r') as f:
        data = json.load(f)

    # 1. Collect all frame indices that need to be extracted
    # We use a set for O(1) lookups and to avoid duplicates if ranges overlap
    indices_to_extract = set()
    range_count = 0
    
    for ann in data.get('annotations', []):
        if 'start_frame' in ann and 'end_frame' in ann:
            # Paper uses 1-based indexing (Frame 1..N)
            # Python/OpenCV uses 0-based indexing (Frame 0..N-1)
            # ACTION: Subtract 1 to align JSON labels with OpenCV frames
            start = int(ann['start_frame']) - 1
            end = int(ann['end_frame']) - 1
            
            # Add all frames in this range to our target set
            for i in range(start, end + 1):
                indices_to_extract.add(i)
            range_count += 1

    if not indices_to_extract:
        print("⚠️ No valid frame ranges found in annotations.")
        return

    print(f"🎯 Target: {len(indices_to_extract)} unique frames from {range_count} ranges.")
    print(f"📸 Extracting frames from {video_path.name}...")

    # 2. Extract specific frames
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Optimization: If the max required frame is larger than video, warn user
    max_idx = max(indices_to_extract)
    if max_idx >= total_frames:
        print(f"⚠️ Warning: Max requested frame ({max_idx}) exceeds video length ({total_frames}). Clipping.")

    extracted_count = 0
    current_frame_idx = 0
    
    # We iterate sequentially because seeking (cap.set) can be slow/imprecise 
    # and action clips are often clustered.
    with tqdm(total=total_frames) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Only save if this frame is in our target set
            if current_frame_idx in indices_to_extract:
                # Filename preserves the native index (0-based) for easy lookup later
                save_path = output_dir / f"frame_{(current_frame_idx + 1):06d}.jpg"
                cv2.imwrite(str(save_path), frame)
                extracted_count += 1
            
            current_frame_idx += 1
            pbar.update(1)

    cap.release()
    print(f"✅ Extraction Complete. Saved {extracted_count} relevant frames to {output_dir.name}/")

# Execute
ANNOTATIONS_PATH = DATA_DIR / "annotations.json"
extract_frames_in_ranges(PROCESSED_VIDEO_PATH, ANNOTATIONS_PATH, FRAMES_DIR)

# Yolo Tracking

In [2]:
def run_yolo_tracking():
    # 1. Load Model
    model_path = MODELS_DIR / "yolo11x-pose.pt"
    if not model_path.exists():
        print("⬇️ Downloading YOLO11x-pose model...")
        urllib.request.urlretrieve(
            "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11x-pose.pt",
            str(model_path)
        )
    
    print("🤖 Loading YOLO11x-Pose...")
    model = YOLO(str(model_path))
    
    # 2. Setup Source
    # We count files for the progress bar, but we do NOT pass the list to the model.
    # Passing the directory path prevents the "Too many open files" error.
    frame_count = len(list(FRAMES_DIR.glob("*.jpg")))
    print(f"🔍 Tracking on {frame_count} frames in {FRAMES_DIR}...")
    
    # 3. Run Tracking
    # FIX: Pass str(FRAMES_DIR) directly. 
    # Ultralytics will sort files alphanumerically (frame_000001 -> frame_000002) automatically.
    results = model.track(source=str(FRAMES_DIR), persist=True, stream=True, verbose=False)
    
    annotations = []
    
    for result in tqdm(results, total=frame_count, desc="Tracking"):
        if result.boxes is None or result.keypoints is None:
            continue

        # Extract frame number from the filename currently being processed
        # result.path returns the full file path, e.g., ".../frame_003314.jpg"
        frame_filename = Path(result.path).stem 
        frame_idx = int(frame_filename.split('_')[1])
        
        # Standardize to Custom 14-point Skeleton (COCO-17 -> 14)
        kpts = result.keypoints.data.cpu().numpy()
        boxes = result.boxes.xyxy.cpu().numpy()
        
        # Handle tracking IDs (might be None if association fails)
        if result.boxes.id is not None:
            ids = result.boxes.id.cpu().numpy()
        else:
            ids = [-1] * len(boxes)
        
        for person_idx, (box, kpt, track_id) in enumerate(zip(boxes, kpts, ids)):
             # COCO 17 to Custom 14 conversion
            # Indices: [0, Neck(avg 5,6), 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
            l_sho, r_sho = kpt[5][:2], kpt[6][:2]
            neck = (l_sho + r_sho) / 2
            neck_conf = min(kpt[5][2], kpt[6][2])
            
            custom_kpts = []
            # Nose (0)
            custom_kpts.extend(kpt[0]) 
            # Neck (New)
            custom_kpts.extend([neck[0], neck[1], neck_conf]) 
            # Remaining COCO indices 5-16
            for idx in range(5, 17):
                custom_kpts.extend(kpt[idx])

            annotations.append({
                "frame_number": frame_idx, # This will now be 1-based (e.g., 3314) matching your file naming
                "track_id": int(track_id),
                "bbox": [float(box[0]), float(box[1]), float(box[2]-box[0]), float(box[3]-box[1])],
                "keypoints": [float(x) for x in custom_kpts],
                "score": float(result.boxes.conf[person_idx])
            })
            
    # Save Tracked JSON
    output_data = {"info": {"fps": 30}, "annotations": annotations}
    with open(TRACKED_JSON, 'w') as f:
        json.dump(output_data, f)
    print(f"✅ Saved tracked annotations to {TRACKED_JSON}")

run_yolo_tracking()

🤖 Loading YOLO11x-Pose...
🔍 Tracking on 18561 frames in d:\Repositories\Boxer-Pose-Estimation\data\DATASET\SixClassBoxingVIDataset\V1\frames_30fps...


Tracking:  29%|██▉       | 5403/18561 [02:52<05:33, 39.46it/s]

WARNING not enough matching points
WARNING not enough matching points


Tracking:  29%|██▉       | 5407/18561 [02:52<05:37, 39.02it/s]

WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points
WARNING not enough matching points


Tracking:  29%|██▉       | 5411/18561 [02:52<05:49, 37.65it/s]

WARNING not enough matching points


Tracking: 100%|██████████| 18561/18561 [11:16<00:00, 27.43it/s]


✅ Saved tracked annotations to d:\Repositories\Boxer-Pose-Estimation\data\DATASET\SixClassBoxingVIDataset\V1\yolo_annotations_tracked.json


# Annotation Filtering

In [3]:
def filter_annotations():
    print("🧹 Cleaning Annotations (MobileNet Filter)...")
    
    classifier_path = PROJECT_ROOT / "models" / "poster_classifier.pth"
    if not classifier_path.exists():
        print("❌ Classifier not found. Skipping filtering.")
        return

    # Load Classifier
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.mobilenet_v2()
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    model.load_state_dict(torch.load(classifier_path, map_location=device, weights_only=True))
    model.eval().to(device)
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    with open(TRACKED_JSON, 'r') as f:
        data = json.load(f)
        
    cleaned_anns = []
    
    # Group by frame
    frames_map = defaultdict(list)
    for ann in data['annotations']:
        frames_map[ann['frame_number']].append(ann)
        
    for frame_num in tqdm(sorted(frames_map.keys())):
        candidates = frames_map[frame_num]
        frame_path = FRAMES_DIR / f"frame_{frame_num:06d}.jpg"
        img = Image.open(frame_path).convert('RGB')
        
        scored_candidates = []
        for ann in candidates:
            x, y, w, h = map(int, ann['bbox'])
            # Pad and crop
            pad = 20
            crop = img.crop((max(0, x-pad), max(0, y-pad), min(img.width, x+w+pad), min(img.height, y+h+pad)))
            
            if crop.size[0] == 0 or crop.size[1] == 0: continue
            
            input_tensor = transform(crop).unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(input_tensor)
                prob = torch.nn.functional.softmax(out, dim=1)[0][0].item() # Index 0 is 'boxer'
                scored_candidates.append((prob, ann))
        
        if scored_candidates:
            # Winner takes all
            best_score, winner = max(scored_candidates, key=lambda x: x[0])
            cleaned_anns.append(winner)
            
    final_data = {"info": data['info'], "annotations": cleaned_anns}
    with open(FINAL_JSON, 'w') as f:
        json.dump(final_data, f)
    print(f"✅ Final Cleaned Dataset Saved: {len(cleaned_anns)} frames.")

filter_annotations()

🧹 Cleaning Annotations (MobileNet Filter)...


100%|██████████| 16156/16156 [08:10<00:00, 32.93it/s]


✅ Final Cleaned Dataset Saved: 16156 frames.


# Clip Visualization

In [7]:
def visualize_sample_clips(num_samples=5):
    # Load Label Annotations (Original Ground Truth)
    gt_path = DATA_DIR / "annotations.json"
    with open(gt_path, 'r') as f:
        gt_data = json.load(f)
        
    # Load Our Computed Poses
    # Ensure this JSON was generated AFTER the YOLO step on the 1-based frames
    with open(FINAL_JSON, 'r') as f:
        pose_data = json.load(f)
    
    # Map frame number -> Pose Annotation
    pose_map = {ann['frame_number']: ann for ann in pose_data['annotations']}
    
    # Filter for valid clips (must have start/end)
    valid_clips = [x for x in gt_data['annotations'] if 'start_frame' in x and 'end_frame' in x]
    
    # Shuffle or just pick first 5
    import random
    samples = random.sample(valid_clips, min(num_samples, len(valid_clips)))
    # samples = valid_clips[:num_samples]
    
    print(f"🎬 Generating {len(samples)} sample clips...")
    
    # Define connections for visualization
    skeleton = [(0,1), (1,2), (1,3), (2,4), (3,5), (4,6), (5,7), (2,8), (3,9), (8,9), (8,10), (9,11), (10,12), (11,13)]
    
    for clip in samples:
        # ---------------------------------------------------------
        # CHANGE: Use labels directly (1-based) to match filenames
        # ---------------------------------------------------------
        start_f = int(clip['start_frame'])
        end_f = int(clip['end_frame'])
        label = clip['class']
        
        # Output filename
        output_filename = CLIPS_DIR / f"{label}_{start_f}_{end_f}.mp4"
        
        # Check first frame to get dimensions
        first_frame_path = FRAMES_DIR / f"frame_{start_f:06d}.jpg"
        if not first_frame_path.exists():
            print(f"⚠️ Start frame {first_frame_path.name} not found. Skipping clip.")
            continue
            
        h, w = cv2.imread(str(first_frame_path)).shape[:2]
        writer = cv2.VideoWriter(str(output_filename), cv2.VideoWriter_fourcc(*'mp4v'), 30, (w, h))
        
        frames_written = 0
        
        for f_idx in range(start_f, end_f + 1):
            img_path = FRAMES_DIR / f"frame_{f_idx:06d}.jpg"
            
            if not img_path.exists():
                # If a frame in the middle is missing, just skip or write black
                continue
            
            frame = cv2.imread(str(img_path))
            
            # Draw Pose if available
            if f_idx in pose_map:
                ann = pose_map[f_idx]
                kpts = ann['keypoints']
                
                # Draw Skeleton Lines
                points = []
                for i in range(0, len(kpts), 3):
                    # kpts structure: [x, y, conf, x, y, conf...]
                    points.append((int(kpts[i]), int(kpts[i+1])))
                    
                for p1, p2 in skeleton:
                    if p1 < len(points) and p2 < len(points):
                        # Optional: Check confidence before drawing
                        # conf1 = kpts[p1*3 + 2]
                        # conf2 = kpts[p2*3 + 2]
                        # if conf1 > 0.5 and conf2 > 0.5:
                        cv2.line(frame, points[p1], points[p2], (0, 255, 0), 2)
                
                # Draw Bounding Box
                if 'bbox' in ann:
                    bx, by, bw, bh = map(int, ann['bbox'])
                    cv2.rectangle(frame, (bx, by), (bx+bw, by+bh), (0, 0, 255), 2)
            
            # Overlay Info
            cv2.putText(frame, f"{label} | Frame: {f_idx}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            
            writer.write(frame)
            frames_written += 1
            
        writer.release()
        print(f"✅ Saved clip: {output_filename.name} ({frames_written} frames)")

visualize_sample_clips(5)

🎬 Generating 5 sample clips...
✅ Saved clip: cross_15501_15510.mp4 (10 frames)
✅ Saved clip: cross_27415_27425.mp4 (11 frames)
✅ Saved clip: cross_4336_4348.mp4 (13 frames)
✅ Saved clip: jab_29307_29320.mp4 (14 frames)
✅ Saved clip: cross_18311_18320.mp4 (10 frames)
